In [8]:
import pandas as pd
import numpy as np
import uuid

# --- 1. DATA SIMULATION SETUP (MOCK IMPORTS) ---
# NOTE: In a real environment, you would load the raw bank data CSV here.
# For demonstration, we'll create a small mock DataFrame to test the logic.

def create_mock_bank_data():
    """
    Creates a small DataFrame simulating raw bank transactions (AMR/BSL)
    to test the Stage 1 logic.
    """
    data = {
        'Transaction ID': [f'TXN{i:05d}' for i in range(1, 11)],
        'Source': ['AMR', 'BSL', 'AMR', 'BSL', 'AMR', 'AMR', 'BSL', 'AMR', 'AMR', 'BSL'],
        'Fund ID': ['50360414'] * 10,
        'Currency': ['USD', 'EUR', 'USD', 'GBP', 'USD', 'EUR', 'USD', 'USD', 'GBP', 'EUR'],
        'Amount': [-10000.00, 500000.00, -500.00, 75000.00, 200.00, -8000.00, 0.00, 1000.00, 50.00, -90000.00],
        'Tran Type': ['TRDL', 'RCPT', 'DOTH', 'TRDL', 'COTH', 'TRDL', 'JRNL', 'RCPT', 'DOTH', 'TRDL'],
        'Comment': [
            'TRANSFER TO WILMINGTON CHARGES 0 00 BEN NEVIS MIDCO LIMITED 2022',
            'TRANSFER FROM GLAS SAS CHARGES 0 00 SATURN BIDCO',
            'TAX FILING FEES Q4 2024', # Non-deal exclusion
            'TRANSFER TO KROLL AGENCY SERVICES LIMITED CHARGES 0 00 GRO BIDCO AS JAN 2021//',
            'FFS_093482 TRANSFER BETWEEN FUNDS', # Inter-fund exclusion
            'TRANSFER TO GLAS EUR VITAPROTECH GROUP CHARGES 30 00 FUNDING VITAPROTECH',
            'INTEREST ACCRUAL ADJUSTMENT', # Journal entry exclusion
            'CC_20250101 CAPITAL CALL', # Capital Call exclusion
            'TRANSFER TO LEGAL ACCT ANNUAL RETAINER', # Non-deal exclusion
            'TRANSFER TO GLAS USD ACQUISITION FUNDING ARROTEX HOLDINGS'
        ],
        # Enrichment Target Fields (Start Blank)
        'Deal ID': [None] * 10,
        'Product ID': [None] * 10,
        'LoanId': [None] * 10,
        'Actual Deal Name': [None] * 10,
        'Event Type': [None] * 10,
        'Enrichment Request status': ['Pending'] * 10
    }
    df = pd.DataFrame(data)
    return df

# --- 2. THE DEAL KNOWLEDGE DIRECTORY ---
# This dictionary simulates the "small directory" you mentioned, representing high-confidence patterns
# and their corresponding Deal IDs, Product IDs, and Loan Facility IDs.

DEAL_KNOWLEDGE_BASE = {
    "BEN NEVIS MIDCO LIMITED 2022": {
        "Deal ID": "D1000007", "Product ID": "PID00006", "LoanId": "D1000007-4"
    },
    "SATURN BIDCO": {
        "Deal ID": "D1000029", "Product ID": "PID00028", "LoanId": "D1000029-2"
    },
    "GRO BIDCO AS": {
        "Deal ID": "D1000015", "Product ID": "PID00014", "LoanId": "D1000015-1"
    },
    "VITAPROTECH": {
        "Deal ID": "D1000040", "Product ID": "PID00039", "LoanId": "D1000040-3"
    },
    "ARROTEX HOLDINGS": {
        "Deal ID": "D1000004", "Product ID": "PID00003", "LoanId": "D1000004-1"
    }
}

# --- 3. STAGE 1 AUTOMATED JOURNAL ENRICHMENT PIPELINE ---

def run_stage_1_enrichment(bank_data_df):
    """
    Executes the three main steps of the Journal Enrichment pipeline:
    1. Exclusion Filtering
    2. Rule-Based Enrichment
    3. ML Enrichment (Placeholder)
    """
    df = bank_data_df.copy()
    initial_pending_count = len(df[df['Enrichment Request status'] == 'Pending'])

    # --------------------------------------------------------------------------
    # STEP 3.1: EXCLUSION FILTERING AND STATUS ASSIGNMENT
    # --------------------------------------------------------------------------
    # Identify transactions that should bypass Cash Control or don't require deal enrichment.
    
    # 3.1.1: Non-Deal (DOTH/COTH) and Journal (JRNL) Exclusions
    exclusion_keywords = [
        'TAX', 'LEGAL', 'FEE', 'RETAINER', 'FFS_', 'CAPITAL CALL', 'DISTRIBUTION'
    ]
    
    # Check for DOTH/COTH/JRNL types OR if comment contains exclusion keywords
    is_exclusion_type = df['Tran Type'].isin(['DOTH', 'COTH', 'JRNL'])
    is_exclusion_comment = df['Comment'].str.upper().str.contains('|'.join(exclusion_keywords), na=False)
    
    # Combine conditions for setting 'Filtered' status
    is_filtered = (is_exclusion_type) | (is_exclusion_comment)

    # Apply Filtering Status
    df.loc[is_filtered, 'Enrichment Request status'] = 'Filtered - No Enrichment Required'
    df.loc[is_filtered, 'Actual Deal Name'] = 'N/A - EXCLUSION'
    df.loc[is_filtered, 'Event Type'] = 'NON_DEAL'

    print(f"-> Filtered {is_filtered.sum()} transactions based on Tran Type/Keywords.")

    # --------------------------------------------------------------------------
    # STEP 3.2: RULE-BASED ENRICHMENT (HIGH-CONFIDENCE MATCHES)
    # --------------------------------------------------------------------------
    # Use the static knowledge base for exact keyword or phrase matching.
    
    # Only process transactions that are still 'Pending'
    pending_mask = df['Enrichment Request status'] == 'Pending'

    def apply_rule_based_matching(row):
        """Iterates through the knowledge base to find a match in the Comment."""
        if row['Enrichment Request status'] != 'Pending':
            return row

        for deal_name, ids in DEAL_KNOWLEDGE_BASE.items():
            if deal_name.upper() in row['Comment'].upper():
                row['Actual Deal Name'] = deal_name
                row['Deal ID'] = ids['Deal ID']
                row['Product ID'] = ids['Product ID']
                row['LoanId'] = ids['LoanId']
                row['Enrichment Request status'] = 'Auto-Enriched - Rule Match'
                row['Event Type'] = 'DRAWDOWN' if row['Amount'] < 0 else 'PAYMENT'
                return row
        
        return row

    # Apply the matching rules only to the pending subset
    df = df.apply(apply_rule_based_matching, axis=1)

    rule_matched_count = len(df[df['Enrichment Request status'] == 'Auto-Enriched - Rule Match'])
    print(f"-> Auto-Enriched {rule_matched_count} transactions using Rule-Based Directory.")

    # --------------------------------------------------------------------------
    # STEP 3.3: ML ENRICHMENT (PLACEHOLDER FOR FUTURE DEVELOPMENT)
    # --------------------------------------------------------------------------
    
    pending_after_rules = len(df[df['Enrichment Request status'] == 'Pending'])
    
    # In a later phase, the ML model (trained on historical data) would analyze the
    # remaining 'Pending' comments and assign the IDs.
    # For now, we simply flag them as needing manual review.
    
    # def run_ml_prediction(df):
    #     # ML model inference logic goes here...
    #     pass 
    
    # df = run_ml_prediction(df) # Assume this runs and assigns some more IDs
    
    print(f"-> {pending_after_rules} transactions remain 'Pending' for ML/Manual Review.")
    
    return df

# --- 4. EXECUTION AND RESULTS DISPLAY ---

# Load or generate the mock data
bank_data_raw = create_mock_bank_data()

# Run the Stage 1 Pipeline
enriched_bank_data = run_stage_1_enrichment(bank_data_raw)

print("\n--- STAGE 1 RESULTS: ENRICHED BANK DATA ---")
print(enriched_bank_data.to_markdown(index=False))

# Calculate the final break counts (for Cash Control Stage)
final_pending_for_stage_2 = enriched_bank_data[enriched_bank_data['Enrichment Request status'].isin(['Pending', 'Filtered - No Enrichment Required'])]

print(f"\nTotal transactions ready for Cash Control (Stage 2): {len(enriched_bank_data)}")
print(f"Total transactions needing Manual Enrichment (True Breaks): {len(enriched_bank_data[enriched_bank_data['Enrichment Request status'] == 'Pending'])}")

# Output the DataFrame as a Markdown CSV
csv_output = enriched_bank_data.to_csv(index=False)


-> Filtered 5 transactions based on Tran Type/Keywords.
-> Auto-Enriched 5 transactions using Rule-Based Directory.
-> 0 transactions remain 'Pending' for ML/Manual Review.

--- STAGE 1 RESULTS: ENRICHED BANK DATA ---
| Transaction ID   | Source   |   Fund ID | Currency   |   Amount | Tran Type   | Comment                                                                        | Deal ID   | Product ID   | LoanId     | Actual Deal Name             | Event Type   | Enrichment Request status         |
|:-----------------|:---------|----------:|:-----------|---------:|:------------|:-------------------------------------------------------------------------------|:----------|:-------------|:-----------|:-----------------------------|:-------------|:----------------------------------|
| TXN00001         | AMR      |  50360414 | USD        |   -10000 | TRDL        | TRANSFER TO WILMINGTON CHARGES 0 00 BEN NEVIS MIDCO LIMITED 2022               | D1000007  | PID00006     | D1000007-4 | BEN NEVIS